# NeuroZip byte-architecture sweep on Kaggle GPUs

This notebook is an orchestration layer. It clones or reuses the public NeuroZip repository, prepares one deterministic raw-byte WikiText-103 split, and trains the configured architectures under the same effective byte budget. Architecture artifacts are persistent: completed runs are detected from `best.pt` plus a complete `summary.json`, while interrupted runs resume from `last.pt` when possible.

The sweep keeps the existing two-layer GRU and LSTM results, then continues with the Transformer, Mamba-Lite, Griffin-Lite, Gated DeltaNet-Lite, and Gated DeltaNet-2-Lite candidates. The Transformer keeps a 2048-token context, uses memory-efficient scaled-dot-product attention where the installed PyTorch runtime supports it, and uses a smaller microbatch plus gradient accumulation to preserve the configured effective batch.

A model is only `PASSED` when decompression is byte-identical and SHA-256-identical. The final recommendation uses actual full-stream BPB, encode/decode speed, and memory relative to the GRU, not validation loss alone.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/Chickaboo/NeuroZip.git'
WORK_ROOT = Path('/kaggle/working/neurozip')
if (WORK_ROOT / '.git').exists():
    print('Reusing source checkout:', WORK_ROOT)
elif not WORK_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORK_ROOT)], check=True)
    print('Cloned source:', REPO_URL)
else:
    raise RuntimeError(f'Existing source path is not a Git checkout: {WORK_ROOT}')
sys.path.insert(0, str(WORK_ROOT / 'src'))
RUN_ENV = os.environ.copy()
RUN_ENV['PYTHONPATH'] = str(WORK_ROOT / 'src') + os.pathsep + RUN_ENV.get('PYTHONPATH', '')
print('Working source:', WORK_ROOT)


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPUs are required; enable Kaggle GPUs before running the sweep.')
GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT < 2:
    raise RuntimeError(f'This experiment expects both Kaggle T4s; only {GPU_COUNT} GPU(s) are visible.')
print('PyTorch:', torch.__version__)
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)])

SWEEP_CONFIG = json.loads((WORK_ROOT / 'configs' / 'architecture_sweep_wikitext103_kaggle.json').read_text())
RUN_ROOT = Path('/kaggle/working/neurozip-byte-architecture-sweep')
DATA_ROOT = RUN_ROOT / 'data'
CACHE_ROOT = Path('/kaggle/working/wikitext-103-cache')
# Keep RUN_ROOT intact: it contains completed architecture checkpoints and
# metrics that must be reused on a later execution. Data preparation is
# deterministic and only refreshes the raw-byte split/manifest.
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

prepare_cmd = [
    sys.executable, '-m', 'neurozip.data.prepare_wikitext',
    '--output-dir', str(DATA_ROOT),
    '--cache-dir', str(CACHE_ROOT),
    '--url', 'https://huggingface.co/datasets/mattdangerw/wikitext-103-raw/resolve/main/wikitext-103-raw-v1.zip?download=true',
    '--train-bytes', str(SWEEP_CONFIG['dataset']['train_bytes']),
    '--valid-bytes', str(SWEEP_CONFIG['dataset']['validation_bytes']),
    '--seed', str(SWEEP_CONFIG['dataset']['seed']),
]
print('Preparing/reusing deterministic data:', ' '.join(prepare_cmd))
subprocess.run(prepare_cmd, cwd=WORK_ROOT, env=RUN_ENV, check=True)
manifest = json.loads((DATA_ROOT / 'manifest.json').read_text())
print(json.dumps(manifest, indent=2, sort_keys=True))


In [ ]:
from neurozip.models.registry import build_model

# Verify the parameter-matching plan before spending GPU time.
parameter_rows = []
for spec in SWEEP_CONFIG['architectures']:
    model = build_model(spec['architecture'], **spec['args'])
    count = sum(parameter.numel() for parameter in model.parameters())
    parameter_rows.append({
        'architecture': spec['name'],
        'label': spec['label'],
        'parameters': count,
        'delta_vs_target': count - SWEEP_CONFIG['matching']['target_parameter_count'],
    })
print(json.dumps(parameter_rows, indent=2))


In [ ]:
import time

ARCHITECTURE_ROOT = RUN_ROOT / 'architectures'
ARCHITECTURE_ROOT.mkdir(parents=True, exist_ok=True)
training = SWEEP_CONFIG['training']
target_batch_per_gpu = int(training['batch_size_per_gpu'])

def run_is_complete(output_dir):
    summary_path = output_dir / 'summary.json'
    best_path = output_dir / 'best.pt'
    if not summary_path.is_file() or not best_path.is_file():
        return False
    try:
        summary = json.loads(summary_path.read_text())
    except (OSError, json.JSONDecodeError):
        return False
    return (
        int(summary.get('total_steps', 0)) >= int(training['steps'])
        and summary.get('best_validation_bpb') is not None
    )

def settings_for(spec):
    override = training.get('architecture_overrides', {}).get(spec['name'], {})
    batch_size = int(override.get('batch_size_per_gpu', target_batch_per_gpu))
    accumulation = int(override.get('gradient_accumulation_steps', training.get('gradient_accumulation_steps', 1)))
    if batch_size <= 0 or accumulation <= 0 or batch_size * accumulation != target_batch_per_gpu:
        raise ValueError(
            f'{spec["name"]}: microbatch {batch_size} * accumulation {accumulation} '
            f'must equal target per-GPU batch {target_batch_per_gpu}'
        )
    amp = str(override.get('amp', training.get('amp', 'off')))
    return batch_size, accumulation, amp

def candidate_settings(spec):
    batch_size, accumulation, amp = settings_for(spec)
    if spec['name'] != 'transformer':
        return [(batch_size, accumulation, amp)]
    # T4 memory varies across Kaggle images. Keep the same effective batch and
    # 2048 context while trying smaller microbatches if the first attempt OOMs.
    candidates = []
    for microbatch in (batch_size, batch_size // 2, batch_size // 4, 1):
        if microbatch >= 1 and target_batch_per_gpu % microbatch == 0:
            candidate = (microbatch, target_batch_per_gpu // microbatch, amp)
            if candidate not in candidates:
                candidates.append(candidate)
    return candidates

def train_command(spec, output_dir, batch_size, accumulation, amp, *, resume, restart):
    command = [
        sys.executable, '-m', 'torch.distributed.run',
        '--standalone', '--nproc-per-node', str(GPU_COUNT),
        '-m', 'neurozip.train',
        '--architecture', spec['architecture'],
        '--train-path', str(DATA_ROOT / 'train.raw'),
        '--valid-path', str(DATA_ROOT / 'validation.raw'),
        '--output-dir', str(output_dir),
        '--steps', str(training['steps']),
        '--batch-size', str(batch_size),
        '--gradient-accumulation-steps', str(accumulation),
        '--amp', amp,
        '--sequence-length', str(SWEEP_CONFIG['representation']['sequence_length']),
        '--learning-rate', str(training['learning_rate']),
        '--weight-decay', str(training['weight_decay']),
        '--gradient-clip', str(training['gradient_clip']),
        '--eval-every', str(training['eval_every']),
        '--validation-eval-bytes', str(training['validation_eval_bytes']),
        '--seed', str(SWEEP_CONFIG['dataset']['seed']),
        '--device', training['device'],
    ]
    if resume:
        command.append('--resume')
    if restart:
        command.append('--restart')
    for key, value in spec['args'].items():
        command.extend(['--' + key.replace('_', '-'), str(value)])
    return command

def write_failure(output_dir, spec, exc, batch_size, accumulation):
    failure = {
        'architecture': spec['name'],
        'failure_reason': f'{type(exc).__name__}: {exc}',
        'batch_size_per_gpu': batch_size,
        'gradient_accumulation_steps': accumulation,
        'time': time.time(),
    }
    (output_dir / 'failure.json').write_text(json.dumps(failure, indent=2, sort_keys=True) + '\n')

skipped_architectures = []
failed_architectures = []
for spec in SWEEP_CONFIG['architectures']:
    name = spec['name']
    output_dir = ARCHITECTURE_ROOT / name
    output_dir.mkdir(parents=True, exist_ok=True)
    if run_is_complete(output_dir):
        skipped_architectures.append(name)
        (output_dir / 'failure.json').unlink(missing_ok=True)
        print(f'\n===== Skipping completed {name}; reusing {output_dir / "best.pt"} =====')
        continue

    last_error = None
    succeeded = False
    for batch_size, accumulation, amp in candidate_settings(spec):
        checkpoint_exists = (output_dir / 'last.pt').is_file() or (output_dir / 'best.pt').is_file()
        # If a resume attempt itself fails (for example because an older
        # checkpoint is corrupt), retry that candidate once from a clean run.
        attempts = [(True, False)] if checkpoint_exists else [(False, True)]
        if checkpoint_exists:
            attempts.append((False, True))
        for resume, restart in attempts:
            command = train_command(
                spec, output_dir, batch_size, accumulation, amp,
                resume=resume, restart=restart,
            )
            print(f'\n===== Training {name}: microbatch={batch_size}, accumulation={accumulation}, amp={amp}, resume={resume} =====')
            print(' '.join(command))
            started = time.perf_counter()
            try:
                subprocess.run(command, cwd=WORK_ROOT, env=RUN_ENV, check=True)
                if run_is_complete(output_dir):
                    (output_dir / 'failure.json').unlink(missing_ok=True)
                    print(f'{name} wall time: {time.perf_counter() - started:.1f}s')
                    succeeded = True
                    break
                raise RuntimeError('trainer exited successfully without a complete summary/checkpoint')
            except Exception as exc:
                last_error = exc
                write_failure(output_dir, spec, exc, batch_size, accumulation)
                print(f'{name} attempt failed; continuing the sweep: {exc}')
        if succeeded:
            break
    if not succeeded:
        failed_architectures.append(name)

print('Skipped completed architectures:', skipped_architectures)
print('Architectures without a completed checkpoint:', failed_architectures)


In [ ]:
BENCHMARK_ROOT = RUN_ROOT / 'benchmark'
benchmark_cmd = [
    sys.executable, '-m', 'neurozip.experiments.architecture_benchmark',
    '--artifacts-root', str(ARCHITECTURE_ROOT),
    '--input', str(DATA_ROOT / 'validation.raw'),
    '--bytes', str(SWEEP_CONFIG['dataset']['heldout_benchmark_bytes']),
    '--output-dir', str(BENCHMARK_ROOT),
    '--device', 'cuda:0',
    '--cdf-bits', str(SWEEP_CONFIG['coding']['cdf_bits']),
    '--expected-architectures',
] + [spec['name'] for spec in SWEEP_CONFIG['architectures']]
print('Benchmarking all retained checkpoints:', ' '.join(benchmark_cmd))
subprocess.run(benchmark_cmd, cwd=WORK_ROOT, env=RUN_ENV, check=True)
comparison = json.loads((BENCHMARK_ROOT / 'comparison.json').read_text())
print('Recommendation:', comparison['recommendation'])


In [ ]:
rows = comparison['results']
try:
    import pandas as pd
    columns = [
        'architecture', 'status', 'train_loss_nats_per_byte', 'train_bpb',
        'validation_loss_nats_per_byte', 'validation_bpb',
        'actual_compressed_bpb', 'payload_bpb', 'compression_ratio',
        'parameter_count', 'checkpoint_bytes', 'training_wall_time_seconds',
        'training_bytes_per_second', 'encode_bytes_per_second',
        'decode_bytes_per_second', 'training_peak_gpu_memory_bytes',
        'training_peak_cpu_memory_bytes', 'encode_peak_gpu_memory_bytes',
        'decode_peak_gpu_memory_bytes', 'peak_process_rss_bytes',
        'byte_identical', 'sha256_identical', 'exact_round_trip',
        'tradeoff_score',
    ]
    display(pd.DataFrame(rows).reindex(columns=columns))
except ImportError:
    print(json.dumps(rows, indent=2, sort_keys=True))

print('\nGeneration sanity checks:')
for row in rows:
    if 'sample_text' in row:
        print(row['architecture'], repr(row['sample_text'][:160]))


In [ ]:
archive_base = Path('/kaggle/working/neurozip-byte-architecture-sweep-artifacts')
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=RUN_ROOT)
print('Download this Kaggle output:', archive_path)
print('Comparison report:', BENCHMARK_ROOT / 'comparison.md')
